This programm runs the calibration for all chosen basins.
- create output folders
- create settingsfiles
- run batch jobs for initialization

In [1]:
# Python-Module für System- und Dateimanagement
import os  # Betriebssystem-Interaktionen
import sys  # Systembezogene Funktionen
from sys import platform  # Erkennen des Betriebssystems
import shutil  # Datei- und Verzeichnisoperationen
import glob  # Suchen von Dateien mit Muster
import stat  # Datei- und Verzeichnisrechte
import time  # Zeitmessung und -steuerung
import datetime  # Arbeiten mit Datum und Uhrzeit
import multiprocessing  # Parallele Verarbeitung
import subprocess  # Ausführen externer Prozesse
from subprocess import Popen, PIPE  # Alternative Möglichkeit für Subprozess-Kommunikation
import pickle  # Speichern und Laden von Python-Objekten (Serialisierung)
import ast  # Abstrakte Syntaxbäume (z. B. Umwandlung von Strings in Python-Objekte)
import configparser  # Einlesen und Schreiben von Konfigurationsdateien
from configparser import ConfigParser  # Alternative Schreibweise

# Numerische Berechnungen & Zufallszahlen
import array  # Arbeit mit Arrays
import random  # Zufallszahlengenerierung
import numpy as np  # Wissenschaftliches Rechnen mit Arrays
from collections import defaultdict, Counter

# Datenverarbeitung & Analyse 
import csv  # Lesen und Schreiben von CSV-Dateien
import xarray as xr  # Arbeiten mit multidimensionalen Arrays (z. B. für NetCDF-Daten)
import cftime  # Arbeiten mit Klimazeitdaten (z. B. NetCDF-Dateien)
import pandas as pd  # Datenanalyse und Tabellenverarbeitung

# Visualisierung
import matplotlib.pyplot as plt  # Erstellen von Diagrammen
import matplotlib.colors as mcolors
import matplotlib.patches as mpatches  

# Geografische Daten & Karten
import cartopy.crs as ccrs  # Koordinatenreferenzsysteme
import cartopy.feature as cfeature  # Geografische Features (z. B. Küstenlinien, Ländergrenzen)
from cartopy.mpl.gridliner import LONGITUDE_FORMATTER, LATITUDE_FORMATTER  # Formatieren von Kartenrastern
import geopandas as gpd  # Verarbeiten von Geodaten, insbesondere Shapefiles
from shapely.geometry import Point  # Arbeiten mit geometrischen Objekten
from osgeo import gdal  # Geodatenformate lesen/schreiben (z. B. Rasterdaten)

# Data paths and names

#### Data path for GRDC file

In [3]:
folder_GRDC_data = '../Step0_preparationGRDCFiles/GRDC_data/GRDC_calibrationStations'
GRDC_file_name = 'GRDC_42calibrationStations_FINAL_LatLonFit.nc'

#### Data path for individual calibration

In [4]:
# Read in data from step 1 initialisation
folder_init_init                 = '../Step1_initialisationFiles/initFiles'  

# Read in data from step 2 individual calibration 
folder_calibr_bash               = 'bashFiles' 
folder_calibr_settings           = 'settingsFiles'  
folder_calibr_settingsConfig     = 'settingsConfigFiles'  
folder_calibr_init               = 'initFiles'  
folder_calibr_output             = 'outputFiles'  

#### Data path for file names

In [5]:
settingsfile_template =  'settings_ccwatm_calibr_template.ini'  
settingsfile = 'settings_ccwatm_calibr_' 

settingsConfigfile_template =  'settingsConfig_ccwatm_calibr_template.txt' 
settingsConfigfile = 'settingsConfig_ccwatm_calibr_'

BASH_runCalibr   = 'BASH_to_run_ccwatm_calibr.sh'
SLURM_runCalibr  = 'SLURM_to_run_ccwatm_calibr.sh' 

calibration_parameters_range_CSV = 'CalibrationScripts/ParamRanges_ccwatm_all7.csv'

# 1. Set default values and define which calibration parameters should be calibrated

In [6]:
# Set default values for calibration parameters (taken from an original settings-files (Status: Jan 2024))
calibration_parameters_dict = {
    'arnoBeta_add':  0.39,
    'alphaDepl': 0.7,
    'recessionCoeff_factor':  5.278,
    'manningsN':  1.86,
    'normalStorageLimit': 0.44,
    'lakeAFactor': 0.33, 
    'lakeEvaFactor': 1.52 }

# Define the list of calibration parameters that should NOT be calibrated / that should be constant equal with the default parameters
# Here all calibration parameters shall be calibrated! This is why 'calibration_parameters_list_default' is empty 
calibration_parameters_list_default = []

# Define the list of calibration parameters, that should be calibrated! 
calibration_parameters_list_all13 = list(calibration_parameters_dict.keys())
calibration_parameters_list = [param for param in calibration_parameters_list_all13 if param not in calibration_parameters_list_default] 
print(calibration_parameters_list)

# NOTE! Only the parameters that are calibrated are allowed to occour in this list! Filtere die Werte basierend auf den aktiven Parametern. 
calibration_parameters_firstRun_all13 = [0.0027, 1.11, 1.28, 4.5, 0.39, 0.7, 2.8 , 5.278, 0.1, 1.86,  0.44 , 0.33, 1.52, 1.0 ] 
calibration_parameters_firstRun = [calibration_parameters_dict[param] for param in calibration_parameters_list] + [1.0] 
print(calibration_parameters_firstRun)

['arnoBeta_add', 'alphaDepl', 'recessionCoeff_factor', 'manningsN', 'normalStorageLimit', 'lakeAFactor', 'lakeEvaFactor']
[0.39, 0.7, 5.278, 1.86, 0.44, 0.33, 1.52, 1.0]


# 2. Define and control the final selection of calibration stations

In [7]:
# Read in GRDC-data-file 
xr_obs_mod = xr.open_dataset(folder_GRDC_data + '/' + GRDC_file_name)

# Read in List of station IDs
list_of_station_IDs = xr_obs_mod['id'].values

# GGf. List of station IDs aus dem GRDC-File manuell einschränken 
list_of_station_IDs_calibration = list_of_station_IDs

print('Total number of stations in GRDC-file: ' + str(len(list_of_station_IDs)))
print('Total number of selected calibration stations: ' + str(len(list_of_station_IDs_calibration)))

Total number of stations in GRDC-file: 42
Total number of selected calibration stations: 42


##  --- no changes necessary below ---

# 3. Automatically create settings-files for the individual calibration process 

#### Automatically delete all entries in DKRZreturn

In [8]:
def empty_folder(folder):
    if not os.path.isdir(folder): return
    for item in os.listdir(folder):
        path = os.path.join(folder, item)
        shutil.rmtree(path) if os.path.isdir(path) else os.remove(path)
    time.sleep(5) 

for f in ["DKRZreturn"]:
    empty_folder(f)

#### Automatically create output folder for all calibration station

In [9]:
# Vorab Output-Ordner erstellen
for ID in list_of_station_IDs_calibration: 
    # Befehlszeile für Erstellen des individuellen Output-Ordner
    mkdir_command = 'mkdir -p ' + folder_calibr_output + '/' + str(int(ID))
    # Befehlszeile ausführen und auf Ergebnis warten
    subprocess.run(mkdir_command, shell=True, stdout=subprocess.PIPE, stderr=subprocess.PIPE)

#### Automatically copy files from Step1_initialisation

In [10]:
# Vorab Dateien vom Step1_initialisation-Folder kopieren
for ID in list_of_station_IDs_calibration: 
    # Befehlszeile zum Kopieren des Step1_initialisation-Folder
    copy_command = 'cp ' + folder_init_init + '/basin_' + str(int(ID)) +  '_*' + ' ' + folder_calibr_init + '/'
    # Befehlszeile ausführen und auf Ergebnis warten
    subprocess.run(copy_command, shell=True, stdout=subprocess.PIPE, stderr=subprocess.PIPE)
    print("Dateien für ID " + str(ID) + " kopiert.")

Dateien für ID 6113050 kopiert.
Dateien für ID 6116200 kopiert.
Dateien für ID 6122260 kopiert.
Dateien für ID 6123400 kopiert.
Dateien für ID 6125100 kopiert.
Dateien für ID 6136145 kopiert.
Dateien für ID 6140400 kopiert.
Dateien für ID 6142150 kopiert.
Dateien für ID 6172050 kopiert.
Dateien für ID 6212740 kopiert.
Dateien für ID 6226800 kopiert.
Dateien für ID 6227510 kopiert.
Dateien für ID 6232911 kopiert.
Dateien für ID 6233410 kopiert.
Dateien für ID 6233510 kopiert.
Dateien für ID 6243050 kopiert.
Dateien für ID 6335240 kopiert.
Dateien für ID 6337200 kopiert.
Dateien für ID 6340160 kopiert.
Dateien für ID 6342501 kopiert.
Dateien für ID 6342900 kopiert.
Dateien für ID 6357500 kopiert.
Dateien für ID 6373304 kopiert.
Dateien für ID 6421100 kopiert.
Dateien für ID 6444600 kopiert.
Dateien für ID 6457870 kopiert.
Dateien für ID 6458450 kopiert.
Dateien für ID 6545050 kopiert.
Dateien für ID 6547110 kopiert.
Dateien für ID 6574150 kopiert.
Dateien für ID 6606655 kopiert.
Dateien 

#### Automatically create settings files for all calibration station (.ini)

In [11]:
# Create dictionary for the file-names of the newly created settingsfiles for calibration
filename_new_setting = {}

# Open the appropriate template, read in data and closes template
setting_file_template =  settingsfile_template
f = open(setting_file_template , "r")
template_xml = f.read()
f.close()

# Automatically create settings-files for each station 
for ID in list_of_station_IDs_calibration: 
    # Duplication from the original settings-file template: 
    template_xml_new = template_xml
    # Modifications from the original settings-file template: Platzhalter '%init_path', '%ini_file' und '%grdc_id' ersetzt
    template_xml_new = template_xml_new.replace('%init_path', folder_calibr_init)
    template_xml_new = template_xml_new.replace('%grdc_id', str(int(ID)))
    # Modifikation from the original settings-file template: Gauges
    lon_fit_list = xr_obs_mod.sel(id=ID).gauged_lon.values
    lat_fit_list = xr_obs_mod.sel(id=ID).gauged_lat.values    
    gauges_str = ' ' + str(lon_fit_list) + ' ' + str(lat_fit_list)
    template_xml_new = template_xml_new.replace('%gauges', gauges_str)
    
    # Modification from the original settings-file template: calibration parameters 
    # Note: In case that any calibration parameters should not be calibrated and instead default values should be read in, than those lines must be activated
    for parameter in calibration_parameters_list_default:
        template_xml_new = template_xml_new.replace('%'+ parameter, str(calibration_parameters_dict[parameter]))
    
    # Create a new settingsfile (duplicated and modified from the template!) with the stations ID ending! 
    filename_new_setting [ID] = folder_calibr_settings + '/' + settingsfile + str(int(ID)) + '.ini'  
    f = open(filename_new_setting [ID], "w+")
    f.write(template_xml_new)
    f.close() 

#### Automatically create master configuration files for calibration process for all calibration station (.txt)

In [12]:
# Create dictionary for the file-names of the newly created masterConfig file
filename_new_masterConfig = {}

# Open the appropriate template, read in data and closes template
f = open(settingsConfigfile_template ,'r')
master_calibr = f.read() 
f.close()
    
# Automatically create master configuration settings-files for each station 
for ID in list_of_station_IDs_calibration: 
    master_calib_new = master_calibr
    # Modifications from the original file template: Platzhalter '%GRDC_file' und '%pfaf_id' ersetzt
    master_calib_new = master_calib_new.replace('%GRDC_file', str(folder_GRDC_data + '/' + GRDC_file_name))
    master_calib_new = master_calib_new.replace('%grdc_id', str(int(ID)))
    # Modifications from the original file template: Platzhalter für Output-Folder ersetzen
    master_calib_new = master_calib_new.replace('%output_path', folder_calibr_output + '/' + str(int(ID)))
    
    # Modifications from the original file template: Platzhalter '%run_model' ersetzen und entweder CCWatM oder CWatM auswählen   
    # !!! Achtung !!! Diese Datei muss sich im Ordner settingsFiles befinden! Das ist etwas blöd, ist nur aufwendig zu ändern - sehr wahrscheinlich im Python-Modul 'CalibrationScripts/calibration_multi_station_netcdf.py'.
    master_calib_new = master_calib_new.replace('%run_model', BASH_runCalibr)   # runpy_CWatM.sh oder runpy_CCWatM.sh 
    
    # Define calibration parameters for first run 
    # NOTE! Only the parameters that are calibrated are allowed to occour in this list! Filtere die Werte basierend auf den aktiven Parametern. 
    calibration_parameters_firstRun_all13 = [0.0027, 1.11, 1.28, 4.5, 0.39, 0.7, 2.8 , 5.278, 0.1, 1.86,  0.44 , 0.33, 1.52, 1.0 ] 
    calibration_parameters_firstRun = [calibration_parameters_dict[param] for param in calibration_parameters_list] + [1.0] 
    # Modifications from the original file template: %calibration_parameters_firstRun'
    master_calib_new = master_calib_new.replace('%calibration_parameters_firstRun', str(calibration_parameters_firstRun))
    # Give the csv-file of calibration parameter ranges that should be used 
    master_calib_new = master_calib_new.replace('%calibration_parameters_range', calibration_parameters_range_CSV)
                                         
    # Create a new settingsfile (duplicated and modified from the template!) with the stations ID ending! 
    filename_new_masterConfig [ID] = folder_calibr_settingsConfig + '/' + settingsConfigfile +  str(int(ID)) + '.txt'
    f = open(filename_new_masterConfig [ID], "w")
    f.write(master_calib_new)
    f.close()

# 4. Automatically run those settings-files for each calibration station with the same SLURM-file 

In [13]:
# Path to the slurm-file 'run_cwatm_slurm-shared.sh', which will be used to run the CWatM model
slurmFile = SLURM_runCalibr

# Automatically run those initialisation files for each station
for ID in list_of_station_IDs_calibration:    
    # Befehlszeile zum Ausführen des SLURM-files
    # Als zusätzliches Argument wird der Name des initialisation files new_setting_file[ID] übergeben. Das von uns geschriebene Bash-file 'run_cwatm_slurm-shared.sh' wartet schon darauf! 
    bash_command = 'sbatch ' + slurmFile + ' ' + filename_new_masterConfig[ID]
    # Befehlszeile ausführen und auf Ergebnis warten
    # subprocess.Popen wird verwendet, subprocess.Popen wird verwendet, um den SLURM-Job im Hintergrund auszuführen, sodass das Jupyter-Notebook nicht blockiert wird und weiterhin ausgeführt werden kann, ohne auf die Fertigstellung des Jobs zu warten.
    # subprocess.Popen is important so that computing can be parallel! Saves a lot of time! 
    result = subprocess.Popen(bash_command, shell=True, stdout=subprocess.PIPE, stderr=subprocess.PIPE)
    output, error = result.communicate() 
    if result.returncode == 0:
        print(f'For station {str(ID)}: {output.decode()} ')
    else:
        print(f'For station {str(ID)}: Error occured {error.decode()} ')


For station 6113050: Submitted batch job 21059228
 
For station 6116200: Submitted batch job 21059229
 
For station 6122260: Submitted batch job 21059230
 
For station 6123400: Submitted batch job 21059231
 
For station 6125100: Submitted batch job 21059232
 
For station 6136145: Submitted batch job 21059233
 
For station 6140400: Submitted batch job 21059234
 
For station 6142150: Submitted batch job 21059235
 
For station 6172050: Submitted batch job 21059236
 
For station 6212740: Submitted batch job 21059237
 
For station 6226800: Submitted batch job 21059238
 
For station 6227510: Submitted batch job 21059239
 
For station 6232911: Submitted batch job 21059240
 
For station 6233410: Submitted batch job 21059241
 
For station 6233510: Submitted batch job 21059242
 
For station 6243050: Submitted batch job 21059243
 
For station 6335240: Submitted batch job 21059244
 
For station 6337200: Submitted batch job 21059245
 
For station 6340160: Submitted batch job 21059247
 
For station 

# 5. Check status of batch jobs
COMPUTING TIME: Really depends on the ressources of DKRZ. Parallel computing of different stations possible. 
<br> COMPUTING TIME: Also really depends on the settings within the "settingsFiles_masterConfig"-Files. 

In [2]:
command = "squeue -u USERID"
result = subprocess.run(command, shell=True, stdout=subprocess.PIPE, stderr=subprocess.PIPE)
if result.returncode == 0:
    print(result.stdout.decode()) 

             JOBID PARTITION     NAME     USER ST       TIME  NODES NODELIST(REASON)
          21154527 interacti spawner-  g300123  R       0:53      1 l40049
          21059273    shared Calibrat  g300123 PD       0:00      1 (QOSMaxCpuPerUserLimit)
          21059272    shared Calibrat  g300123 PD       0:00      1 (QOSMaxCpuPerUserLimit)
          21059271    shared Calibrat  g300123 PD       0:00      1 (QOSMaxCpuPerUserLimit)
          21059270    shared Calibrat  g300123 PD       0:00      1 (QOSMaxCpuPerUserLimit)
          21059269    shared Calibrat  g300123 PD       0:00      1 (QOSMaxCpuPerUserLimit)
          21059268    shared Calibrat  g300123 PD       0:00      1 (QOSMaxCpuPerUserLimit)
          21059267    shared Calibrat  g300123 PD       0:00      1 (QOSMaxCpuPerUserLimit)
          21059266    shared Calibrat  g300123 PD       0:00      1 (QOSMaxCpuPerUserLimit)
          21059264    shared Calibrat  g300123 PD       0:00      1 (QOSMaxCpuPerUserLimit)
          21